In [ ]:
import os
import cv2
import subprocess
import shutil # Import shutil for directory operations

zip_path = "/content/archive (21).zip"
extract_path = "extracted_images"

# Ensure the extraction directory is clean
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
os.makedirs(extract_path, exist_ok=True)

# Extract the ZIP file using the system's unzip command
try:
    subprocess.run(["unzip", zip_path, "-d", extract_path], check=True)
    print(f"Successfully extracted {zip_path} to {extract_path}")
except subprocess.CalledProcessError as e:
    print(f"Error extracting zip file: {e}")
    print(f"Stderr: {e.stderr}")
    # Exit or handle the error appropriately if extraction fails
    exit()

# Read all images
images = []

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(tuple([".jpg", ".jpeg", ".png", ".bmp"])): # Use a tuple for suffix check
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is not None:
                images.append(img)

print("Total images loaded:", len(images))


Error extracting zip file: Command '['unzip', '/content/archive (21).zip', '-d', 'extracted_images']' returned non-zero exit status 9.
Stderr: None
Total images loaded: 0


In [ ]:
for root, dirs, files in os.walk(extract_path):
    print(root)

extracted_images


In [ ]:
dataset_path = "extracted_images/brain_tumor_dataset"

In [ ]:
print("Yes:", len(os.listdir("extracted_images/brain_tumor_dataset/yes")))
print("No:", len(os.listdir("extracted_images/brain_tumor_dataset/no")))

FileNotFoundError: [Errno 2] No such file or directory: 'extracted_images/brain_tumor_dataset/yes'

In [ ]:
import numpy as np

data = []
labels = []

dataset_path = "extracted_images/brain_tumor_dataset"

for category in ["yes", "no"]:

    folder = os.path.join(dataset_path, category)

    for img_name in os.listdir(folder):

        img_path = os.path.join(folder, img_name)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is not None:

            img = cv2.resize(img, (64, 64))

            data.append(img.flatten())

            if category == "yes":
                labels.append(1)
            else:
                labels.append(0)

X = np.array(data)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy * 100, "%")

In [ ]:
for i in range(10):
    actual = "Yes" if y_test[i] == 1 else "No"
    predicted = "Yes" if y_pred[i] == 1 else "No"

    print(f"Image {i+1}")
    print("Actual:", actual)
    print("Predicted:", predicted)
    print("-"*20)

In [ ]:
from google.colab import files
import cv2
import numpy as np

# Upload image
uploaded = files.upload()

# Get uploaded filename
filename = list(uploaded.keys())[0]

# Read image
img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

# Same preprocessing used during training
img = cv2.resize(img, (64, 64))
img = img.flatten().reshape(1, -1)

# Predict
prediction = model.predict(img)

if prediction[0] == 1:
    print("Yes")
else:
    print("No")